In [0]:

from pyspark.sql import functions as F

# ============================================================
# DAY 3 - COMPLETE GOLD LAYER VALIDATION
# Catalog: capretailnew
# Schema: gold
# Read-only validation - no tables are modified
# ============================================================

GOLD = "capretailnew.gold"
SILVER = "capretailnew.silver"

# ------------------------------------------------------------
# 1. Load all Gold and required Silver tables
# ------------------------------------------------------------

gold_dim_product = spark.table(GOLD + ".gold_dim_product")
gold_dim_store = spark.table(GOLD + ".gold_dim_store")
gold_dim_supplier = spark.table(GOLD + ".gold_dim_supplier")

gold_fact_sales = spark.table(GOLD + ".gold_fact_sales")
gold_fact_returns = spark.table(GOLD + ".gold_fact_returns")
gold_fact_inventory = spark.table(GOLD + ".gold_fact_inventory_movements")

gold_inventory_ledger = spark.table(GOLD + ".gold_inventory_movement_ledger")
gold_daily_sales = spark.table(GOLD + ".gold_daily_sales_summary")
gold_store_kpis = spark.table(GOLD + ".gold_store_sales_kpis")

silver_sales = spark.table(SILVER + ".silver_sales_order_lines")
silver_returns = spark.table(SILVER + ".silver_sales_returns")
silver_inventory = spark.table(SILVER + ".silver_inventory_movements")

silver_products = spark.table(SILVER + ".silver_product_master")
silver_stores = spark.table(SILVER + ".silver_store_master")
silver_suppliers = spark.table(SILVER + ".silver_supplier_master")

results = []

def record_check(name, actual, expected, condition=None):
    passed = actual == expected if condition is None else condition
    results.append((name, actual, expected, "PASS" if passed else "FAIL"))

def count_rows(df):
    return df.count()

def duplicate_groups(df, key):
    return (
        df.groupBy(key)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

def null_count(df, key):
    return df.filter(
        F.col(key).isNull() |
        (F.trim(F.col(key).cast("string")) == "")
    ).count()

def unmatched_count(child, parent, key):
    parent_keys = parent.select(key).distinct()
    return child.join(parent_keys, on=key, how="left_anti").count()

# ------------------------------------------------------------
# 2. Verify all nine Gold table record counts
# ------------------------------------------------------------

print("\n========== 1. GOLD TABLE RECORD COUNTS ==========")

expected_counts = [
    ("Gold Product Dimension", gold_dim_product, 139),
    ("Gold Store Dimension", gold_dim_store, 148),
    ("Gold Supplier Dimension", gold_dim_supplier, 157),
    ("Gold Sales Fact", gold_fact_sales, 11319),
    ("Gold Returns Fact", gold_fact_returns, 316),
    ("Gold Inventory Fact", gold_fact_inventory, 634),
    ("Gold Inventory Ledger", gold_inventory_ledger, 634)
]

for name, df, expected in expected_counts:
    actual = count_rows(df)
    record_check(name, actual, expected)
    print(name, "Expected:", expected, "| Actual:", actual)

# ------------------------------------------------------------
# 3. Verify Gold dimension primary keys
# ------------------------------------------------------------

print("\n========== 2. DIMENSION PRIMARY KEY CHECKS ==========")

dimension_checks = [
    ("Product Dimension", gold_dim_product, "product_id"),
    ("Store Dimension", gold_dim_store, "store_id"),
    ("Supplier Dimension", gold_dim_supplier, "supplier_id")
]

for name, df, key in dimension_checks:
    nulls = null_count(df, key)
    duplicates = duplicate_groups(df, key)

    record_check(name + " Null Keys", nulls, 0)
    record_check(name + " Duplicate Key Groups", duplicates, 0)

    print(name, "| Null keys:", nulls, "| Duplicate groups:", duplicates)

# ------------------------------------------------------------
# 4. Verify fact table primary/business keys
# ------------------------------------------------------------

print("\n========== 3. FACT KEY CHECKS ==========")

fact_checks = [
    ("Sales Fact", gold_fact_sales, "order_line_id"),
    ("Returns Fact", gold_fact_returns, "return_id"),
    ("Inventory Fact", gold_fact_inventory, "movement_id"),
    ("Inventory Ledger", gold_inventory_ledger, "movement_id")
]

for name, df, key in fact_checks:
    nulls = null_count(df, key)
    duplicates = duplicate_groups(df, key)

    record_check(name + " Null Keys", nulls, 0)
    record_check(name + " Duplicate Key Groups", duplicates, 0)

    print(name, "| Null keys:", nulls, "| Duplicate groups:", duplicates)

# ------------------------------------------------------------
# 5. Verify required Sales status and calculated columns
# ------------------------------------------------------------

print("\n========== 4. SALES DATA QUALITY ==========")

sales_null_status = null_count(gold_fact_sales, "order_status")

record_check("Gold Sales Missing Order Status", sales_null_status, 0)

print("Sales records with NULL/blank order_status:", sales_null_status)

sales_amount_columns = [
    "gross_sales",
    "discount_amount",
    "sales_after_discount_before_tax",
    "sales_including_tax_before_refunds"
]

for column_name in sales_amount_columns:
    missing = gold_fact_sales.filter(
        F.col(column_name).isNull()
    ).count()

    record_check("Sales NULL " + column_name, missing, 0)
    print("Missing", column_name, ":", missing)

# ------------------------------------------------------------
# 6. Verify foreign-key relationships
# ------------------------------------------------------------

print("\n========== 5. FOREIGN KEY CHECKS ==========")

fk_checks = [
    ("Sales -> Product", gold_fact_sales, gold_dim_product, "product_id"),
    ("Sales -> Store", gold_fact_sales, gold_dim_store, "store_id"),

    ("Returns -> Product", gold_fact_returns, gold_dim_product, "product_id"),
    ("Returns -> Store", gold_fact_returns, gold_dim_store, "store_id"),

    ("Inventory -> Product", gold_fact_inventory, gold_dim_product, "product_id"),
    ("Inventory -> Store", gold_fact_inventory, gold_dim_store, "store_id"),

    ("Product -> Supplier", gold_dim_product, gold_dim_supplier, "supplier_id")
]

for name, child, parent, key in fk_checks:
    unmatched = unmatched_count(child, parent, key)

    record_check(name + " Unmatched Records", unmatched, 0)

    print(name, "| Unmatched records:", unmatched)

# ------------------------------------------------------------
# 7. Verify fact-to-Silver row count preservation
# ------------------------------------------------------------

print("\n========== 6. SILVER TO GOLD RECONCILIATION ==========")

reconciliation = [
    ("Sales", silver_sales, gold_fact_sales, 11319),
    ("Returns", silver_returns, gold_fact_returns, 316),
    ("Inventory", silver_inventory, gold_fact_inventory, 634)
]

for name, silver_df, gold_df, expected in reconciliation:
    silver_count = silver_df.count()
    gold_count = gold_df.count()

    record_check(name + " Silver Count", silver_count, expected)
    record_check(name + " Gold Count", gold_count, expected)
    record_check(name + " Row Preservation", gold_count, silver_count)

    print(
        name,
        "| Silver:", silver_count,
        "| Gold:", gold_count,
        "| Difference:", silver_count - gold_count
    )

# ------------------------------------------------------------
# 8. Verify returns linked to original Sales
# ------------------------------------------------------------

print("\n========== 7. RETURNS LINK VALIDATION ==========")

if "sales_link_status" in gold_fact_returns.columns:
    display(
        gold_fact_returns
        .groupBy("sales_link_status")
        .count()
        .orderBy("sales_link_status")
    )

if "original_order_id" in gold_fact_returns.columns:
    missing_sales_links = gold_fact_returns.filter(
        F.col("original_order_id").isNull()
    ).count()

    print("Returns without matched original Sales:", missing_sales_links)
    print("Expected based on Silver investigation: 3")

    record_check(
        "Returns without matched original Sales",
        missing_sales_links,
        3
    )

# ------------------------------------------------------------
# 9. Verify business summary tables
# ------------------------------------------------------------

print("\n========== 8. BUSINESS SUMMARY VALIDATION ==========")

daily_count = gold_daily_sales.count()
store_kpi_count = gold_store_kpis.count()

print("Daily Sales Summary records:", daily_count)
print("Store Sales KPI records:", store_kpi_count)

daily_duplicates = (
    gold_daily_sales
    .groupBy("sales_date", "store_id", "category")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

store_kpi_duplicates = duplicate_groups(
    gold_store_kpis, "store_id"
)

record_check("Daily Summary Duplicate Groups", daily_duplicates, 0)
record_check("Store KPI Duplicate Groups", store_kpi_duplicates, 0)

print("Daily Summary Duplicate Groups:", daily_duplicates)
print("Store KPI Duplicate Groups:", store_kpi_duplicates)

# Verify daily summary totals against Sales Fact
daily_totals = gold_daily_sales.agg(
    F.sum("total_order_lines").alias("lines"),
    F.sum("total_quantity_sold").alias("quantity"),
    F.sum("gross_sales").alias("gross"),
    F.sum("total_discount").alias("discount"),
    F.sum("sales_after_discount_before_tax").alias("net")
).first()

fact_totals = gold_fact_sales.agg(
    F.count("order_line_id").alias("lines"),
    F.sum("quantity").alias("quantity"),
    F.sum("gross_sales").alias("gross"),
    F.sum("discount_amount").alias("discount"),
    F.sum("sales_after_discount_before_tax").alias("net")
).first()

def close_enough(a, b):
    if a is None or b is None:
        return a == b
    return abs(float(a) - float(b)) < 0.02

summary_comparisons = [
    ("Daily Summary Order Lines", daily_totals["lines"], fact_totals["lines"]),
    ("Daily Summary Quantity", daily_totals["quantity"], fact_totals["quantity"]),
    ("Daily Summary Gross Sales", daily_totals["gross"], fact_totals["gross"]),
    ("Daily Summary Discount", daily_totals["discount"], fact_totals["discount"]),
    ("Daily Summary Net Sales", daily_totals["net"], fact_totals["net"])
]

for name, actual, expected in summary_comparisons:
    record_check(name, actual, expected, close_enough(actual, expected))
    print(name, "| Summary:", actual, "| Fact:", expected)

# Verify Store KPI totals against Sales Fact
store_totals = gold_store_kpis.agg(
    F.sum("total_order_lines").alias("lines"),
    F.sum("total_quantity_sold").alias("quantity"),
    F.sum("gross_sales").alias("gross"),
    F.sum("total_discount").alias("discount"),
    F.sum("total_sales_after_discount").alias("net")
).first()

store_comparisons = [
    ("Store KPI Order Lines", store_totals["lines"], fact_totals["lines"]),
    ("Store KPI Quantity", store_totals["quantity"], fact_totals["quantity"]),
    ("Store KPI Gross Sales", store_totals["gross"], fact_totals["gross"]),
    ("Store KPI Discount", store_totals["discount"], fact_totals["discount"]),
    ("Store KPI Net Sales", store_totals["net"], fact_totals["net"])
]

for name, actual, expected in store_comparisons:
    record_check(name, actual, expected, close_enough(actual, expected))
    print(name, "| KPI:", actual, "| Fact:", expected)

# ------------------------------------------------------------
# 10. Verify inventory ledger
# ------------------------------------------------------------

print("\n========== 9. INVENTORY LEDGER VALIDATION ==========")

ledger_missing_sequence = gold_inventory_ledger.filter(
    F.col("movement_sequence").isNull()
).count()

ledger_missing_cumulative = gold_inventory_ledger.filter(
    F.col("cumulative_quantity_change").isNull()
).count()

record_check("Ledger Missing Sequence", ledger_missing_sequence, 0)
record_check("Ledger Missing Cumulative Value", ledger_missing_cumulative, 0)

print("Missing movement sequence:", ledger_missing_sequence)
print("Missing cumulative quantity:", ledger_missing_cumulative)

# ------------------------------------------------------------
# 11. Final result
# ------------------------------------------------------------

print("\n========== 10. FINAL GOLD VALIDATION REPORT ==========")

print(
    "{:<48} {:<30} {:<20} {}".format(
        "CHECK", "ACTUAL", "EXPECTED", "STATUS"
    )
)

print("-" * 115)

for name, actual, expected, status in results:
    print(
        "{:<48} {:<30} {:<20} {}".format(
            name, str(actual), str(expected), status
        )
    )

passed = sum(1 for row in results if row[3] == "PASS")
failed = sum(1 for row in results if row[3] == "FAIL")

print("\nTotal Checks:", len(results))
print("Passed:", passed)
print("Failed:", failed)

if failed == 0:
    print("\nALL GOLD VALIDATION CHECKS PASSED")
else:
    print("\nGOLD VALIDATION NEEDS ATTENTION")
    print("Review the FAIL rows above before starting Day 4.")


In [0]:

# ============================================================
# DAY 4 - AUTOMATED DATA QUALITY TEST FRAMEWORK
# Retail Sales and Inventory Optimization
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType
)

# 1. Verify that existing validation results are available
if 'results' not in globals() or not results:
    results = []  # Initialize results as an empty list or load actual validation results here.
    raise Exception(
        "Validation results not found. Run the Gold validation cell first."
    )

# 2. Convert the 57 checks into a structured test matrix
test_rows = []

for index, result in enumerate(results, start=1):

    test_name = result[0]
    actual_value = result[1]
    expected_value = result[2]
    test_status = result[3]

    test_rows.append((
        "DQ" + str(index).zfill(3),
        str(test_name),
        str(actual_value),
        str(expected_value),
        str(test_status)
    ))

# 3. Define test matrix schema
test_schema = StructType([
    StructField("test_id", StringType(), False),
    StructField("test_description", StringType(), False),
    StructField("actual_result", StringType(), True),
    StructField("expected_result", StringType(), True),
    StructField("test_status", StringType(), False)
])

# 4. Create test matrix DataFrame
test_matrix = spark.createDataFrame(
    test_rows,
    schema=test_schema
)

# 5. Add execution timestamp
test_matrix = test_matrix.withColumn(
    "test_execution_timestamp",
    F.current_timestamp()
)

# 6. Display complete test matrix
print("========== AUTOMATED DATA QUALITY TEST MATRIX ==========")

display(
    test_matrix.orderBy("test_id")
)

# 7. Calculate test summary
total_tests = len(test_rows)

passed_tests = sum(
    1 for row in test_rows if row[4] == "PASS"
)

failed_tests = sum(
    1 for row in test_rows if row[4] == "FAIL"
)

print("\n========== TEST EXECUTION SUMMARY ==========")
print("Total Tests :", total_tests)
print("Passed      :", passed_tests)
print("Failed      :", failed_tests)

# 8. Persist test results in Delta
test_table = "capretailnew.retail.dq_test_results"

(
    test_matrix.write
    .format("delta")
    .mode("append")
    .saveAsTable(test_table)
)

print("\nTest results saved to:", test_table)

# 9. Fail the workflow if any test fails
if failed_tests > 0:
    raise Exception(
        str(failed_tests) +
        " data quality tests failed. Check the test matrix."
    )

print("\nALL AUTOMATED DATA QUALITY TESTS PASSED")
